In [25]:
import sqlite3
import pandas as pd

claims = pd.read_csv('/Users/ada/myprojects/my-first-app/data/claims.csv')
plans = pd.read_csv('/Users/ada/myprojects/my-first-app/data/plans.csv')

# Inspect claims
print("Claims Info:")
claims.info()
#print("\nClaims Head:")
#print(claims.head())

# Inspect plans
print("\n\nPlans Info:")
plans.info()
print("\nPlans Head:")
print(plans.head())

def clean_and_load_data():
    print("Starting data pipeline...")

    # 1. LOAD RAW DATA
    df_plans = pd.read_csv("/Users/ada/myprojects/my-first-app/data/plans.csv")
    df_claims = pd.read_csv("/Users/ada/myprojects/my-first-app/data/claims.csv")
    
    # 2. STANDARDIZE COLUMN NAMES (lowercase & stripped of spaces)
    df_plans.columns = df_plans.columns.str.strip().str.lower()
    df_claims.columns = df_claims.columns.str.strip().str.lower()
    
    # 3. DEDUPLICATE RECORDS
    df_plans = df_plans.drop_duplicates()
    df_claims = df_claims.drop_duplicates()
    
    # 4. CLEAN STRING TEXT FIELDS (remove accidental padding spaces)
    for col in df_plans.select_dtypes(include=["object"]).columns:
        df_plans[col] = df_plans[col].astype(str).str.strip()
        
    for col in df_claims.select_dtypes(include=["object"]).columns:
        df_claims[col] = df_claims[col].astype(str).str.strip()
        
    # 5. CLEAN CURRENCY FIELDS (strip '$' and ',' signs, convert to numeric float)
    if "monthly_premium" in df_plans.columns:
        df_plans["monthly_premium"] = df_plans["monthly_premium"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False)
        df_plans["monthly_premium"] = pd.to_numeric(df_plans["monthly_premium"], errors="coerce")
        
    if "claim_amount" in df_claims.columns:
        df_claims["claim_amount"] = df_claims["claim_amount"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False)
        df_claims["claim_amount"] = pd.to_numeric(df_claims["claim_amount"], errors="coerce")
        
    # 6. STANDARDIZE DATE FORMATS
    if "claim_date" in df_claims.columns:
        df_claims["claim_date"] = pd.to_datetime(df_claims["claim_date"], errors="coerce")
        # Convert back to standard ISO string format (YYYY-MM-DD) for SQLite compatibility
        df_claims["claim_date"] = df_claims["claim_date"].dt.strftime("%Y-%m-%d")

    # 7. FILL EMPTY OR NULL CELLS (Handle remaining missing data safely)
    df_plans = df_plans.fillna({"monthly_premium": 0.0, "deductible": 0})
    df_claims = df_claims.fillna({"claim_amount": 0.0, "status": "Unknown"})

    # 8. EXPORT TO SQLITE
    with sqlite3.connect("coverage.db") as conn:
        df_plans.to_sql(name="plans", con=conn, if_exists="replace", index=False)
        df_claims.to_sql(name="claims", con=conn, if_exists="replace", index=False)
        
    print("Success: Cleaned data successfully written to coverage.db!")

clean_and_load_data()

#TESTING QUERIES
# 1. Define your 5 target SQL queries in a dictionary layout
queries = {
    "1. Simple Filter: What's the deductible on the Gold PPO plan?": """
SELECT annual_deductible 
FROM plans 
WHERE LOWER(plan_name) = 'gold ppo';
""",
    "2. Aggregation: How many claims are pending for member M1001?": """
SELECT COUNT(*) AS pending_count 
FROM claims 
WHERE member_id = 'M1001' 
  AND LOWER(status) = 'pending';
""",
    "3. Numerical Comparison: Which plans have a monthly premium under $400?": """
SELECT plan_name, monthly_premium 
FROM plans 
WHERE monthly_premium < 400 
ORDER BY monthly_premium DESC;
""",
    "4. Table Relation: Relational JOIN between claims and plans": """
SELECT 
    c.claim_id, 
    c.member_id, 
    p.plan_name, 
    c.claim_amount 
FROM claims c 
JOIN plans p ON c.plan_id = p.plan_id;
""",
    "5. Top-N Analysis: Most claimed medical procedures": """
SELECT 
    procedure, 
    COUNT(*) AS claim_count, 
    SUM(claim_amount) AS total_claimed_dollars
FROM claims 
GROUP BY procedure 
ORDER BY claim_count DESC 
LIMIT 5;
"""
}

# 2. Open database connection, fetch results, and construct Markdown layout string
md_content = "# Structured Queries and Outputs Reference\n\n"

with sqlite3.connect("coverage.db") as conn:
    for title, sql_query in queries.items():
        # Execute query using pandas
        df_res = pd.read_sql_query(sql_query, conn)
        
        # Format markdown blocks
        md_content += f"## {title}\n\n"
        md_content += "### SQL Query\n```sql" + sql_query + "```\n\n"
        md_content += "### Output Table\n" + df_res.to_markdown(index=False) + "\n\n"
        md_content += "---\n\n"

# 3. Save the built layout block explicitly onto your file disk
with open("structured_queries.md", "w") as f:
    f.write(md_content)

print("File 'structured_queries.md' generated successfully in your working folder!")

Claims Info:
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   claim_id      5 non-null      str  
 1   member_id     5 non-null      str  
 2   plan_id       5 non-null      str  
 3   procedure     5 non-null      str  
 4   claim_amount  5 non-null      int64
 5   status        5 non-null      str  
 6   date_filed    5 non-null      str  
dtypes: int64(1), str(6)
memory usage: 412.0 bytes


Plans Info:
<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   plan_id            3 non-null      str  
 1   plan_name          3 non-null      str  
 2   monthly_premium    3 non-null      int64
 3   annual_deductible  3 non-null      int64
 4   copay_pct          3 non-null      int64
 5   coverage_type      3 non-null      str  
 6   ne

/var/folders/px/sh49d3y54n136p2rffhtwj740000gn/T/ipykernel_23271/2125264016.py:35: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_plans.select_dtypes(include=["object"]).columns:
/var/folders/px/sh49d3y54n136p2rffhtwj740000gn/T/ipykernel_23271/2125264016.py:38: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See ht